# Notebook baseline — Gemma-3 4B (4-bit) sur 5 cas

> ⚠️ Prototype pédagogique. Non destiné au diagnostic.

Reproductibilité : charge la baseline et exécute l'inférence sur 5 cas de `data/eval/`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import GEMMA3_ID, LABELS_CSV
from src.models import load_model, free_model, vram_used_gb
from src.inference import run_single
from src.preprocessing import preprocess
from src.prompts import BASELINE
from src.postprocess import parse_json, apply_uncertainty_rule
import csv, json

In [ ]:
with open(LABELS_CSV, newline='', encoding='utf-8') as f:
    cases = list(csv.DictReader(f))[:5]
len(cases)

In [ ]:
model, processor = load_model(GEMMA3_ID)
print('VRAM:', vram_used_gb(), 'Go')

In [ ]:
for c in cases:
    p = ROOT / c['file']
    img, qc = preprocess(p)
    raw, lat = run_single(model, processor, img, BASELINE)
    parsed, valid = parse_json(raw)
    final, reason = apply_uncertainty_rule(parsed, valid, qc['image_quality'])
    print(f"{c['case_id']} GT={c['ground_truth']} pred={final} ({reason}) {lat}ms")

In [ ]:
free_model(model, processor)